# 03 — Cleaning (Scrub)

Takes the raw scraped strings from `data/raw/books_raw.csv` and converts them into an analysis-ready dataset saved to `data/processed/books_clean.csv`.

**Principle**: nothing is dropped silently. Every removal, conversion or flag is counted and reported, so the transformation from raw to clean is fully auditable.

In [1]:
import sys
from pathlib import Path
import re

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np

raw = pd.read_csv("../data/raw/books_raw.csv")
print(raw.shape)
raw.dtypes

(1000, 13)


title                        str
price_raw                    str
availability_raw             str
star_rating_word             str
detail_url                   str
category                     str
description                  str
upc                          str
price_excl_tax_raw           str
price_incl_tax_raw           str
tax_raw                      str
availability_detail_raw      str
num_reviews_raw            int64
dtype: object

## Step 1 — survey the raw data

Before writing any cleaning logic, check which columns actually carry information. A column with a single repeated value can't contribute to analysis and should be identified now rather than discovered halfway through the exploration.

In [2]:
for col in raw.columns:
    n_unique = raw[col].nunique(dropna=False)
    n_missing = raw[col].isna().sum()
    print(f"{col:28} unique={n_unique:5}  missing={n_missing}")

title                        unique=  999  missing=0
price_raw                    unique=  903  missing=0
availability_raw             unique=    1  missing=0
star_rating_word             unique=    5  missing=0
detail_url                   unique= 1000  missing=0
category                     unique=   50  missing=0
description                  unique=  999  missing=2
upc                          unique= 1000  missing=0
price_excl_tax_raw           unique=  903  missing=0
price_incl_tax_raw           unique=  903  missing=0
tax_raw                      unique=    1  missing=0
availability_detail_raw      unique=   21  missing=0
num_reviews_raw              unique=    1  missing=0


### Findings from the survey

| Column | Unique | Verdict |
|---|---|---|
| `availability_raw` | 1 | Constant (`In stock`) — no information, drop |
| `tax_raw` | 1 | Constant (`£0.00`) — no information, drop |
| `num_reviews_raw` | 1 | Constant (`0`) — the site records no reviews at all, drop |
| `price_excl_tax_raw` | 903 | Identical to `price_raw` (tax is always zero) — redundant, drop |
| `price_incl_tax_raw` | 903 | Identical to `price_raw` — redundant, drop |
| `upc` | 1000 | Fully unique → the real primary key |
| `title` | 999 | One duplicated title, with different UPCs |
| `description` | 998 + 2 missing | Two books genuinely have no description on the site |
| `availability_detail_raw` | 21 | Real variation in stock counts — the usable availability signal |
| `category` | 50 | Key categorical variable for the analysis |

Five of thirteen columns carry zero information. Verifying this *before* the exploration
stage avoids producing meaningless charts of constant values later.

In [3]:
same_excl = (raw["price_raw"] == raw["price_excl_tax_raw"]).all()
same_incl = (raw["price_raw"] == raw["price_incl_tax_raw"]).all()
print("price_raw == price_excl_tax_raw:", same_excl)
print("price_raw == price_incl_tax_raw:", same_incl)
print("tax values:", raw["tax_raw"].unique())
print("availability values:", raw["availability_raw"].unique())
print("review values:", raw["num_reviews_raw"].unique())

price_raw == price_excl_tax_raw: True
price_raw == price_incl_tax_raw: True
tax values: <StringArray>
['£0.00']
Length: 1, dtype: str
availability values: <StringArray>
['In stock']
Length: 1, dtype: str
review values: [0]


In [4]:
print("Books with no description:")
print(raw.loc[raw["description"].isna(), ["title", "category", "detail_url"]])

dup_titles = raw[raw["title"].duplicated(keep=False)]
print("\nDuplicated title rows:")
print(dup_titles[["title", "upc", "price_raw", "category"]])

Books with no description:
                                                 title  category  \
160  The Bridge to Consciousness: I'm Writing the B...   Default   
995  Alice in Wonderland (Alice's Adventures in Won...  Classics   

                                            detail_url  
160  https://books.toscrape.com/catalogue/the-bridg...  
995  https://books.toscrape.com/catalogue/alice-in-...  

Duplicated title rows:
                      title               upc price_raw category
236  The Star-Touched Queen  1528279aec1f3dce    £46.02  Fantasy
358  The Star-Touched Queen  4a7a25be293ad678    £32.30  Fantasy


### Duplicate and missing-value decisions

**Duplicate title** — "The Star-Touched Queen" appears twice, but with different UPCs
(`1528279aec1f3dce` / `4a7a25be293ad678`) and different prices (£46.02 / £32.30).
These are two separate catalogue entries, not a scraping artefact. **Both rows are kept.**
`upc` is used as the primary key rather than `title`.

**Missing descriptions** — two books have no description on the site itself. Since
description is not used in the quantitative analysis, these rows are kept and the field
is left as `NaN` rather than imputed with a placeholder. **No rows are dropped.**

Final row count after cleaning: still 1000.

## Step 2 — regex cleaning

Three fields need parsing out of text:

| Field | Raw | Target |
|---|---|---|
| price | `£51.77` | `51.77` (float) |
| stock | `In stock (22 available)` | `22` (int) |
| rating | `Three` | `3` (int) |

Price and stock use regex; the rating is a fixed vocabulary of five words, so a lookup
dictionary is both clearer and safer than a pattern.

In [5]:
def parse_price(raw_price: str) -> float:
    """'£51.77' -> 51.77"""
    match = re.search(r"(\d+\.\d+)", raw_price)
    return float(match.group(1)) if match else np.nan


def parse_stock(raw_availability: str) -> int:
    """'In stock (22 available)' -> 22"""
    match = re.search(r"\((\d+)\s+available\)", raw_availability)
    return int(match.group(1)) if match else np.nan


RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def parse_rating(word: str) -> int:
    """'Three' -> 3"""
    return RATING_MAP.get(word, np.nan)

In [6]:
print(parse_price("£51.77"), parse_price("£9.99"))
print(parse_stock("In stock (22 available)"), parse_stock("In stock (1 available)"))
print(parse_rating("Three"), parse_rating("Five"), parse_rating("Unknown"))

51.77 9.99
22 1
3 5 nan


## Step 3 — build the clean DataFrame

Rather than mutating the raw frame in place, a new frame is constructed explicitly.

This makes the raw → clean transformation readable as a single block: every column in the output has a visible origin, and the raw data stays untouched in memory for comparison.

In [7]:
clean = pd.DataFrame({
    "upc": raw["upc"],
    "title": raw["title"],
    "category": raw["category"],
    "price": raw["price_raw"].apply(parse_price),
    "rating": raw["star_rating_word"].apply(parse_rating),
    "stock": raw["availability_detail_raw"].apply(parse_stock),
    "description": raw["description"],
    "detail_url": raw["detail_url"],
})

print(clean.shape)
clean.dtypes

(1000, 8)


upc                str
title              str
category           str
price          float64
rating           int64
stock            int64
description        str
detail_url         str
dtype: object

In [8]:
print("Rows:", len(clean))
print("\nNulls after parsing:")
print(clean[["price", "rating", "stock"]].isna().sum())

print("\nValue ranges:")
print("price :", clean['price'].min(), "→", clean['price'].max())
print("rating:", clean['rating'].min(), "→", clean['rating'].max())
print("stock :", clean['stock'].min(), "→", clean['stock'].max())

print("\nRating distribution:")
print(clean["rating"].value_counts().sort_index())

Rows: 1000

Nulls after parsing:
price     0
rating    0
stock     0
dtype: int64

Value ranges:
price : 10.0 → 59.99
rating: 1 → 5
stock : 1 → 22

Rating distribution:
rating
1    226
2    196
3    203
4    179
5    196
Name: count, dtype: int64


## Step 4 — NumPy sanity statistics

A first quantitative look at the numeric columns, using NumPy directly on the underlying arrays. This is a validation step, not the analysis — the goal is to confirm the values are plausible before building anything on top of them.

In [9]:
prices = clean["price"].to_numpy()
ratings = clean["rating"].to_numpy()
stocks = clean["stock"].to_numpy()

def describe(name, arr):
    print(f"{name}")
    print(f"  n      : {arr.size}")
    print(f"  mean   : {np.mean(arr):.2f}")
    print(f"  median : {np.median(arr):.2f}")
    print(f"  std    : {np.std(arr, ddof=1):.2f}")
    print(f"  min/max: {np.min(arr):.2f} / {np.max(arr):.2f}")
    print(f"  q1/q3  : {np.percentile(arr, 25):.2f} / {np.percentile(arr, 75):.2f}")
    print()

describe("PRICE", prices)
describe("RATING", ratings)
describe("STOCK", stocks)

PRICE
  n      : 1000
  mean   : 35.07
  median : 35.98
  std    : 14.45
  min/max: 10.00 / 59.99
  q1/q3  : 22.11 / 47.46

RATING
  n      : 1000
  mean   : 2.92
  median : 3.00
  std    : 1.43
  min/max: 1.00 / 5.00
  q1/q3  : 2.00 / 4.00

STOCK
  n      : 1000
  mean   : 8.59
  median : 7.00
  std    : 5.65
  min/max: 1.00 / 22.00
  q1/q3  : 3.00 / 14.00



## Conclusions — what the cleaning stage established

**Structural findings**

- 5 of 13 scraped columns carried zero information (`availability_raw`, `tax_raw`,
  `num_reviews_raw`, and two redundant price columns identical to `price_raw`).
  Dropped with justification rather than silently.
- `upc` is the true primary key — 1000 unique values. `title` is not (999 unique).
- No rows were dropped. Final dataset: **1000 rows × 8 columns**, the same row count
  we started with.

**Parsing outcome**

All three regex/lookup parsers succeeded on all 1000 rows — zero nulls introduced in
`price`, `rating` or `stock`. The resulting ranges are all plausible:
price £10.00–£59.99, rating 1–5, stock 1–22.

**The critical finding: this data is synthetic, and the statistics prove it**

The site warns that prices and ratings are randomly assigned. The descriptive statistics
confirm it quantitatively:

| Evidence | Observation | Implication |
|---|---|---|
| Price mean ≈ median | 35.07 vs 35.98 | Symmetric, no skew |
| Price quartile gaps | 22.11 → 35.98 → 47.46 (≈12–13 apart) | Evenly spaced = uniform distribution |
| Price bounds | exactly £10.00 – £59.99 | Hard limits of a random range |
| Rating counts | 226 / 196 / 203 / 179 / 196 | Near-uniform (expected 200 each) |
| Stock | mean 8.59 > median 7.00, range 1–22 | Mild right skew, consistent with random draws |

Real-world book prices are right-skewed and cluster around common price points; real
ratings skew high (most books rated 4–5). Neither pattern appears here.

**Consequence for the exploration stage**: correlations between price, rating and stock are
expected to be approximately zero. The exploration notebook therefore treats this as a
**methodological exercise on a known-random dataset** — the techniques (distributions,
correlation, boxplots, outlier detection) are applied and interpreted correctly, and the
absence of relationships is reported as the honest result rather than over-interpreted into
findings that do not exist.

## Export

Saved in two formats:

- **CSV** — the working format for the next notebook. Compact, universally readable.
- **JSON** — records-oriented, preserving dtypes and nested-friendly structure.
  Demonstrates the alternative serialisation format covered in the course, and is the
  format an API or downstream service would typically consume.

In [10]:
clean.to_csv("../data/processed/books_clean.csv", index=False)
clean.to_json("../data/processed/books_clean.json", orient="records", indent=2)

print("CSV :", Path("../data/processed/books_clean.csv").stat().st_size / 1024, "KB")
print("JSON:", Path("../data/processed/books_clean.json").stat().st_size / 1024, "KB")

CSV : 1576.9375 KB
JSON: 1726.91796875 KB


In [11]:
check_csv = pd.read_csv("../data/processed/books_clean.csv")
check_json = pd.read_json("../data/processed/books_clean.json")

print("CSV shape :", check_csv.shape)
print("JSON shape:", check_json.shape)
print("Dtypes match:", (check_csv.dtypes == clean.dtypes).all())

CSV shape : (1000, 8)
JSON shape: (1000, 8)
Dtypes match: True
